# Notebook 06: CFG Scale 扫描（class-conditional MNIST）

**目标**：训一个 class-conditional DDPM on MNIST，扫描 CFG scale 看其对生成的影响。

**关键观察**：
- CFG=0：等价于无条件采样
- CFG≈1：精确条件采样
- CFG>1：超条件化（更"典型"但可能伪影）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
T = 1000
NULL_CLASS = 10  # 11th class as null token
P_UNCOND = 0.1   # conditional dropout

In [ ]:
# Schedule
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1 - betas
ac = alphas.cumprod(0)

# MNIST
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
ds = datasets.MNIST('./data', train=True, download=True, transform=tf)
loader = DataLoader(ds, batch_size=128, shuffle=True, num_workers=2)
print(f'MNIST size: {len(ds)}')

In [ ]:
# 简化的 class-conditional UNet（用 conv 替代以演示）
class CondUNet(nn.Module):
    def __init__(self, ch=32, num_classes=11):
        super().__init__()
        self.t_emb = nn.Embedding(T, 64)
        self.y_emb = nn.Embedding(num_classes, 64)
        self.proj = nn.Linear(128, ch*4)

        # 简化：用 conv + AdaGN-style 注入
        self.enc1 = nn.Conv2d(1, ch, 3, padding=1)
        self.enc2 = nn.Conv2d(ch, ch*2, 3, padding=1, stride=2)
        self.enc3 = nn.Conv2d(ch*2, ch*4, 3, padding=1, stride=2)
        self.mid = nn.Conv2d(ch*4, ch*4, 3, padding=1)
        self.dec3 = nn.Conv2d(ch*4*2, ch*2, 3, padding=1)
        self.dec2 = nn.Conv2d(ch*2*2, ch, 3, padding=1)
        self.dec1 = nn.Conv2d(ch*2, 1, 3, padding=1)
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
    def forward(self, x, t, y):
        emb = torch.cat([self.t_emb(t), self.y_emb(y)], dim=-1)
        h_emb = self.proj(F.silu(emb)).unsqueeze(-1).unsqueeze(-1)
        h1 = F.silu(self.enc1(x))
        h2 = F.silu(self.enc2(h1))
        h3 = F.silu(self.enc3(h2))
        h = F.silu(self.mid(h3) + h_emb)
        d3 = F.silu(self.dec3(torch.cat([self.up(h)[:, :, :h2.shape[2], :h2.shape[3]], h2], dim=1)))
        d2 = F.silu(self.dec2(torch.cat([self.up(d3)[:, :, :h1.shape[2], :h1.shape[3]], h1], dim=1)))
        return self.dec1(torch.cat([d2, h1], dim=1))

model = CondUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-4)
print(f'params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M')

## 训练 - 含 conditional dropout (CFG 训练)

注：这是教学演示，2-3 epoch 即可看到效果。完整训练需要更久。

In [ ]:
import time
model.train()
EPOCHS = 3  # 教学用，实际建议 10+
for ep in range(EPOCHS):
    t0 = time.time(); losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        # Conditional dropout
        mask = torch.rand(x.shape[0], device=device) < P_UNCOND
        y_in = y.clone(); y_in[mask] = NULL_CLASS

        t = torch.randint(0, T, (x.shape[0],), device=device)
        eps = torch.randn_like(x)
        xt = ac[t].view(-1,1,1,1).sqrt() * x + (1-ac[t]).view(-1,1,1,1).sqrt() * eps
        loss = F.mse_loss(model(xt, t, y_in), eps)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    print(f'epoch {ep+1}: loss={np.mean(losses):.4f}, time={time.time()-t0:.0f}s')

## CFG 采样函数

In [ ]:
@torch.no_grad()
def cfg_sample(class_label, cfg_scale=3.0, n=8, n_steps=50):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    y_cond = torch.full((n,), class_label, device=device, dtype=torch.long)
    y_uncond = torch.full((n,), NULL_CLASS, device=device, dtype=torch.long)
    step = T // n_steps
    timesteps = list(range(0, T, step))[::-1]
    for i, t in enumerate(timesteps):
        t_prev = timesteps[i+1] if i+1 < len(timesteps) else -1
        t_tensor = torch.full((n,), t, device=device, dtype=torch.long)
        eps_cond = model(x, t_tensor, y_cond)
        eps_uncond = model(x, t_tensor, y_uncond)
        eps = eps_uncond + cfg_scale * (eps_cond - eps_uncond)
        x0_hat = ((x - (1-ac[t]).sqrt() * eps) / ac[t].sqrt()).clamp(-1, 1)
        if t_prev < 0:
            x = x0_hat
        else:
            ac_prev = ac[t_prev]
            x = ac_prev.sqrt() * x0_hat + (1 - ac_prev).sqrt() * eps
    return x

## 实验：CFG scale 扫描

In [ ]:
cfg_scales = [0.0, 1.0, 3.0, 7.5, 15.0]
class_label = 7  # generate digit '7'
n = 8

fig, axes = plt.subplots(len(cfg_scales), n, figsize=(n*1, len(cfg_scales)*1))
for i, cfg in enumerate(cfg_scales):
    torch.manual_seed(42)
    samples = cfg_sample(class_label, cfg_scale=cfg, n=n, n_steps=50)
    for j in range(n):
        img = (samples[j, 0].cpu() / 2 + 0.5).clamp(0,1).numpy()
        axes[i][j].imshow(img, cmap='gray'); axes[i][j].axis('off')
        if j == 0: axes[i][j].set_ylabel(f'CFG={cfg}', rotation=0, ha='right', va='center', size=8)
plt.suptitle(f'Generated digit {class_label} at different CFG scales')
plt.tight_layout(); plt.show()

## 实验 2: 不同 class label

In [ ]:
fig, axes = plt.subplots(10, 5, figsize=(5, 10))
for cls in range(10):
    torch.manual_seed(cls)
    samples = cfg_sample(cls, cfg_scale=3.0, n=5, n_steps=50)
    for j in range(5):
        img = (samples[j, 0].cpu() / 2 + 0.5).clamp(0,1).numpy()
        axes[cls][j].imshow(img, cmap='gray'); axes[cls][j].axis('off')
    axes[cls][0].set_ylabel(f'class {cls}', rotation=0, ha='right', va='center')
plt.suptitle('CFG=3.0, all 10 classes')
plt.tight_layout(); plt.show()

## 观察与思考

1. **CFG=0**：生成应近似无条件（任意数字混合）。是否符合？
2. **CFG=1**：理论上等价于真实 $p(x|y)$。生成图像最自然。
3. **CFG>>1**：图像变得"过度规范化"——是否看到数字变"过分典型"？
4. 在 MNIST 这种简单数据上，CFG 的作用没有 SD 那么戏剧化。原因是什么？
5. （进阶）实现 negative class（用其他 class 作为 negative），观察排斥效应